# 🏋️ Exercise 2 — Transform
**Tujuan:** membersihkan 5 masalah kualitas data dan membuat kolom `total_amount`.

> Isi bagian bertanda `# TODO`, lalu jalankan sel **Cek Jawaban**.

## ⚙️ Persiapan
Pastikan kernel **.venv** sudah dipilih (pojok kanan atas) dan file `.env` sudah diisi, lalu jalankan sel-sel berikut.

In [ ]:
# ==== KONFIGURASI ====
# Kredensial database dibaca dari file .env di root repo (salin .env.example → .env, isi dari kartu akses).
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

GITHUB_USER = "AisahAlfiyatusR"     # akun GitHub pemateri (repo asal yang Anda fork)
REPO        = "data-science-workshop"
RAW_BASE    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
API_URL     = os.getenv("API_URL") or RAW_BASE + "/api/static/sales_page_{page}.json"

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = int(os.getenv("DB_PORT", 5432))
DB_NAME     = os.getenv("DB_NAME", "workshop")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA      = DB_USER   # setiap peserta punya schema pribadi bernama sama dengan username

if not (DB_HOST and DB_USER and DB_PASSWORD):
    print("⚠️ File .env belum ditemukan/diisi — bagian database tidak akan berjalan.")
else:
    print(f"Konfigurasi OK — login database sebagai {DB_USER} @ {DB_HOST}")

In [ ]:
import os, requests
import pandas as pd
from sqlalchemy import create_engine, text, Date
from sqlalchemy.engine import URL
pd.set_option("display.max_columns", 20)


def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"

## Data mentah (sudah disiapkan)

In [ ]:
def extract_sales(api_url, page_size=100):
    """Mengambil seluruh halaman data dari API."""
    rows, page = [], 1
    while True:
        resp = requests.get(page_url(api_url, page, page_size), timeout=30)
        resp.raise_for_status()                      # error jika status bukan 2xx
        payload = resp.json()
        rows.extend(payload["data"])
        print(f"Halaman {page}: {len(payload['data'])} baris")
        if not payload.get("has_next"):
            break
        page += 1
    return pd.DataFrame(rows)


def load_csv_fallback(filename):
    """Cadangan jika API gagal: baca CSV dari folder data/ (clone lokal) atau langsung dari GitHub."""
    for p in (filename, f"data/{filename}", f"../data/{filename}"):
        if os.path.exists(p):
            return pd.read_csv(p, dtype=str, keep_default_na=False)
    url = f"{RAW_BASE}/data/{filename}"
    print("Membaca cadangan dari GitHub:", url)
    return pd.read_csv(url, dtype=str, keep_default_na=False)

In [ ]:
try:
    df_raw = extract_sales(API_URL)
except Exception as e:
    print("⚠️ API tidak dapat diakses:", e)
    df_raw = load_csv_fallback("sales_raw.csv")

print("Total baris:", len(df_raw))
df_raw.head()

## Tugas
Lengkapi `transform_sales()`:
1. Hapus duplikat
2. Ubah `transaction_date` menjadi tanggal (ada format `YYYY-MM-DD` **dan** `DD/MM/YYYY`)
3. Rapikan `city` (hapus spasi, huruf kapital di awal kata)
4. `quantity` → integer; nilai kosong diisi **1**
5. `unit_price` → angka; nilai negatif dijadikan positif
6. Tambahkan `total_amount = quantity * unit_price`

In [ ]:
def transform_sales(df):
    df = df.copy()
    # TODO 1: hapus duplikat

    # TODO 2: konversi tanggal (petunjuk: pd.to_datetime(..., format=..., errors="coerce") dua kali, lalu fillna)

    # TODO 3: rapikan kota

    # TODO 4: quantity

    # TODO 5: unit_price

    # TODO 6: total_amount

    return df.reset_index(drop=True)

df_clean = transform_sales(df_raw)
df_clean.head()

## Cek Jawaban

In [ ]:
import datetime as dt
assert len(df_clean) == 300, f"Jumlah baris {len(df_clean)}, seharusnya 300 (duplikat sudah dihapus?)"
assert all(isinstance(x, (dt.date, pd.Timestamp)) for x in df_clean["transaction_date"]), "Tanggal belum dikonversi"
assert pd.to_datetime(df_clean["transaction_date"]).max() <= pd.Timestamp("2026-03-31"), "Ada tanggal tertukar hari/bulan"
assert set(df_clean["city"]) == {"Malang", "Surabaya", "Batu", "Sidoarjo", "Pasuruan"}, f"Kota: {set(df_clean['city'])}"
assert str(df_clean["quantity"].dtype).startswith("int"), "quantity harus integer"
assert (df_clean["unit_price"] >= 0).all(), "Masih ada harga negatif"
assert int(df_clean["total_amount"].sum()) == 146_350_000, "total_amount belum tepat"
print("✅ Exercise 2 benar! Total revenue Q1 = Rp", f"{df_clean['total_amount'].sum():,.0f}")

## Diskusi
Apakah mengisi `quantity` kosong dengan 1 selalu tepat? Apa alternatifnya (hapus baris, tanyakan ke pemilik data, tandai dengan kolom flag)? Tuliskan pendapat Anda di bawah.

*Jawaban diskusi:* ...